In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd

# Ensure the notebook can find the /src directory
root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.append(str(root_path))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from src.data_loader import VesselDataLoader
from src.visualizer import plot_block_space, plot_mode_statistics
from src.data_processing import engineer_telemetry_features
from src.mission_profiler import MissionProfiler

In [ ]:
filenames = ["Rotherhithe_voy_179.csv", "Wembley_voy_236.csv"]

all_registry_entries = []
all_global_stats = []

profiler = MissionProfiler(speed_threshold=1.0)

for filename in filenames:
    print(f"Processing {filename}...")
    
    # 1. Load and process
    raw_data_file = root_path / "data" / "raw" / filename
    loader = VesselDataLoader(raw_data_file)
    raw_df = loader.load_and_clean()
    processed_df = engineer_telemetry_features(raw_df, filter_method='savgol')
    
    # 2. Classify and compute
    modes_df = profiler.classify_modes(processed_df)
    
    # Registry needs to be generated per file to capture specific metadata
    registry_df = profiler.generate_block_registry(
        modes_df, 
        source_file_name=filename, 
        merge_loitering=True, 
        unify_port_ops=True
    )
    all_registry_entries.append(registry_df)
    
    stats = profiler.extract_global_statistics(modes_df)
    all_global_stats.append(stats)

combined_registry = pd.concat(all_registry_entries, ignore_index=True)
combined_stats = pd.concat(all_global_stats).groupby(level=0).sum()

display(combined_registry.head())

,Source_File,Start_Time,Duration_h,Energy_kWh,Mean_Power_kW,H2_Rate_Lower_kg_h,H2_Rate_Upper_kg_h,Relative_Fatigue_Activity_Rate,Mean_Power_Fluctuation_Intensity,Stay_ID,MODE,Loitering_Handling,Port_Handling
0,Rotherhithe_voy_179.csv,2025-12-22 10:50,103.666667,42999.118700,414.782495,22.633553,27.663232,0.137294,5.196725,2,Sea_Transit_Ballast,Merged,NaN
1,Rotherhithe_voy_179.csv,2025-12-26 18:30,26.666667,12651.439397,474.428977,25.888300,31.641255,0.108792,3.934591,3,Port_Loading,NaN,Unified
2,Rotherhithe_voy_179.csv,2025-12-27 21:10,60.416667,29626.396332,490.367939,26.758045,32.704278,0.015265,0.603777,4,Sea_Transit_Laden,Merged,NaN
3,Rotherhithe_voy_179.csv,2025-12-30 09:35,32.666667,35029.920012,1072.344490,58.514924,71.518240,0.653432,18.209712,5,Port_Unloading,NaN,Unified


In [ ]:
fig_bricks_fatigue = plot_block_space(combined_registry, y_axis_metric='Relative_Fatigue_Activity_Rate')
fig_bricks_fatigue.show()

fig_stats_fatigue = plot_mode_statistics(combined_stats, y_axis_metric='Relative_Fatigue_Activity_Rate')
fig_stats_fatigue.show()